<a href="https://colab.research.google.com/github/nandakumar3261/ai-mentor-portfolio-Nandakumar/blob/main/Day2_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [ ]:
import os
from google import genai
from pydantic import BaseModel, ValidationError
from typing import List


# =========================================
# Pydantic Models
# =========================================

class MCQ(BaseModel):
    question: str
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    answer: str


class FlutterQuiz(BaseModel):
    total_questions: int
    questions: List[MCQ]


# =========================================
# Gemini Client
# =========================================

client = genai.Client(
    api_key=os.environ['GEMINI_API_KEY']
)


# =========================================
# File Path
# =========================================

FILE_PATH = "/content/sample_data/flutter_mcq.rtf"


# =========================================
# Read Uploaded File
# =========================================

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("File loaded successfully")


# =========================================
# Extract MCQ Function
# =========================================

def extract_flutter_mcqs(
    raw_text: str,
    max_retries: int = 1
) -> FlutterQuiz:

    prompt = f"""
    Extract all Flutter MCQs from this text.

    Return ONLY valid JSON.

    JSON format rules:
    {{
      "total_questions": number,
      "questions": [
        {{
          "question": "",
          "option_a": "",
          "option_b": "",
          "option_c": "",
          "option_d": "",
          "answer": ""
        }}
      ]
    }}

    Text:
    {raw_text}
    """

    for attempt in range(max_retries + 1):

        try:

            resp = client.models.generate_content(
                model='gemini-2.5-flash',

                contents=prompt,

                config={
                    'response_mime_type': 'application/json',

                    'response_schema':
                    FlutterQuiz.model_json_schema(),
                },
            )

            return FlutterQuiz.model_validate_json(
                resp.text
            )

        except ValidationError as e:

            if attempt == max_retries:
                raise

            print("Validation Error. Retrying...")

            fix_prompt = f"""
            Fix this JSON to match schema.

            Validation Error:
            {e}

            Original JSON:
            {resp.text}
            """

            resp = client.models.generate_content(
                model='gemini-2.5-flash',

                contents=fix_prompt,

                config={
                    'response_mime_type': 'application/json',

                    'response_schema':
                    FlutterQuiz.model_json_schema(),
                },
            )

            return FlutterQuiz.model_validate_json(
                resp.text
            )


# =========================================
# Extract Quiz
# =========================================

quiz = extract_flutter_mcqs(raw_text)


# =========================================
# Display Questions
# =========================================

# print(f"\nExtracted {quiz.total_questions} MCQs")


# for i, q in enumerate(quiz.questions):

#     print(f"\nQuestion {i+1}")
#     print(q.question)

#     print(f"a) {q.option_a}")
#     print(f"b) {q.option_b}")
#     print(f"c) {q.option_c}")
#     print(f"d) {q.option_d}")

#     print(f"Answer: {q.answer}")


# =========================================
# Display JSON Output
# =========================================

print("\n===================================")
print("JSON OUTPUT")
print("===================================\n")

json_output = quiz.model_dump_json(indent=2)

print(json_output)

File loaded successfully

JSON OUTPUT

{
  "total_questions": 20,
  "questions": [
    {
      "question": "What is Flutter?",
      "option_a": "Database",
      "option_b": "Programming Language",
      "option_c": "UI Toolkit",
      "option_d": "Operating System",
      "answer": "c"
    },
    {
      "question": "Which language is primarily used in Flutter?",
      "option_a": "Java",
      "option_b": "Kotlin",
      "option_c": "Dart",
      "option_d": "Python",
      "answer": "c"
    },
    {
      "question": "Who developed Flutter?",
      "option_a": "Microsoft",
      "option_b": "Apple",
      "option_c": "Google",
      "option_d": "Facebook",
      "answer": "c"
    },
    {
      "question": "Which widget is immutable in Flutter?",
      "option_a": "StatefulWidget",
      "option_b": "StatelessWidget",
      "option_c": "Container",
      "option_d": "Scaffold",
      "answer": "b"
    },
    {
      "question": "Which widget is used for arranging widgets vertically